# PHOENIX Training Pipeline Guide

This notebook shows how to use the general training pipeline from code. The demo uses `sklearn.datasets.load_breast_cancer()` directly in memory, so no demo dataset file needs to be saved into the repository.

## 1. CLI Usage

From `ai-ml/training_pipeline`, run:

```powershell
run_pipeline.bat
```

The runner uses the `phoenix` conda environment, starts TensorBoard in a separate CLI, then launches training.

In [1]:
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from src import TrainingPipeline

In [2]:
ROOT = Path.cwd()
DEFAULT_CONFIG = ROOT / "configs" / "default_config.yaml"

## 2. Build an In-Memory Demo Dataset

This is a small common binary-classification dataset that trains quickly and is suitable for notebook demos.

In [3]:
breast_cancer = load_breast_cancer(as_frame=True)
demo_df = breast_cancer.frame.copy() # type: ignore
demo_df["target"] = breast_cancer.target # type: ignore

demo_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 3. Run a Short Demo Training Job

This example uses the class API, feeds the dataset directly from memory, enables TensorBoard, and trains a small PyTorch MLP for two epochs.

In [4]:
pipeline = TrainingPipeline(config_path=DEFAULT_CONFIG)
pipeline.set_dataset_frame(
    dataframe=demo_df,
    target_column="target",
    dataset_name="breast_cancer_demo",
)
pipeline.set_model(
    model_type="pytorch_mlp",
    hyperparameters={
        "input_dim": demo_df.shape[1] - 1,
        "hidden_dim": 32,
        "output_dim": 2,
    },
)
pipeline.set_training(epochs=30, batch_size=32, learning_rate=0.001, verbose=True)
pipeline.enable_tensorboard(True, log_dir="logs/tensorboard")
pipeline.set_output(
    checkpoint_prefix="breast_cancer_demo",
    previous_checkpoints_to_keep=1,
)

result = pipeline.run(run_id="breast_cancer_demo_run")
result["reports"]

2026-04-25 00:35:57,992 | INFO | training_pipeline | CONFIG SNAPSHOT
2026-04-25 00:35:57,993 | INFO | training_pipeline | CONFIG | section=dataset | value={"path": "data/raw/example.csv", "abnormal_path": "", "target_column": "target", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}
2026-04-25 00:35:57,994 | INFO | training_pipeline | CONFIG | section=preprocessing | value={"missing_value_strategy": "mean", "normalization": "standard", "encoding": "none", "feature_selection": false, "selected_features": []}
2026-04-25 00:35:57,995 | INFO | training_pipeline | CONFIG | section=model | value={"type": "pytorch_mlp", "task_type": "classification", "hyperparameters": {"input_dim": 30, "hidden_dim": 32, "output_dim": 2}, "name": ""}
2026-04-25 00:35:57,996 | INFO | training_pipeline | CONFIG | section=training | value={"batch_size": 32, "epochs": 30, "learning_rate": 0.001, "verbose": true, "tensorboard_enabled": true, "tensorboard_log_dir": "l

[train] model=simple_mlp type=pytorch epochs=30 batch_size=32
[train] epoch=001/030 | loss=0.6568 | val_accuracy=0.8941 | val_precision=0.8548 | val_recall=1.0000 | val_f1=0.9217 | val_auc=0.9693 | best=updated
[train] epoch=002/030 | loss=0.5495 | val_accuracy=0.9529 | val_precision=0.9298 | val_recall=1.0000 | val_f1=0.9636 | val_auc=0.9858 | best=updated
[train] epoch=003/030 | loss=0.4119 | val_accuracy=0.9529 | val_precision=0.9455 | val_recall=0.9811 | val_f1=0.9630 | val_auc=0.9900
[train] epoch=004/030 | loss=0.2755 | val_accuracy=0.9529 | val_precision=0.9623 | val_recall=0.9623 | val_f1=0.9623 | val_auc=0.9923
[train] epoch=005/030 | loss=0.1828 | val_accuracy=0.9765 | val_precision=0.9811 | val_recall=0.9811 | val_f1=0.9811 | val_auc=0.9947 | best=updated
[train] epoch=006/030 | loss=0.1344 | val_accuracy=0.9765 | val_precision=0.9811 | val_recall=0.9811 | val_f1=0.9811 | val_auc=0.9988
[train] epoch=007/030 | loss=0.1025 | val_accuracy=0.9882 | val_precision=1.0000 | val_re

2026-04-25 00:36:00,711 | INFO | training_pipeline | Training complete | model_name=simple_mlp | model_type=pytorch | task_type=classification


[train] epoch=024/030 | loss=0.0324 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=025/030 | loss=0.0326 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=026/030 | loss=0.0301 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=027/030 | loss=0.0444 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=028/030 | loss=0.0281 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=029/030 | loss=0.0278 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000
[train] epoch=030/030 | loss=0.0260 | val_accuracy=0.9765 | val_precision=1.0000 | val_recall=0.9623 | val_f1=0.9808 | val_auc=1.0000


2026-04-25 00:36:00,739 | INFO | training_pipeline | METRIC | metric=val_accuracy | value=0.9764705882352941
2026-04-25 00:36:00,740 | INFO | training_pipeline | METRIC | metric=val_precision | value=1.0
2026-04-25 00:36:00,741 | INFO | training_pipeline | METRIC | metric=val_recall | value=0.9622641509433962
2026-04-25 00:36:00,741 | INFO | training_pipeline | METRIC | metric=val_f1 | value=0.9807692307692307
2026-04-25 00:36:00,742 | INFO | training_pipeline | METRIC | metric=val_confusion_matrix | value=[[32, 0], [2, 51]]
2026-04-25 00:36:00,743 | INFO | training_pipeline | METRIC | metric=val_auc | value=1.0
2026-04-25 00:36:00,743 | INFO | training_pipeline | METRIC | metric=test_accuracy | value=0.9651162790697675
2026-04-25 00:36:00,743 | INFO | training_pipeline | METRIC | metric=test_precision | value=0.9811320754716981
2026-04-25 00:36:00,743 | INFO | training_pipeline | METRIC | metric=test_recall | value=0.9629629629629629
2026-04-25 00:36:00,747 | INFO | training_pipeline 

[eval] validation: accuracy=0.9765 | precision=1.0000 | recall=0.9623 | f1=0.9808 | auc=1.0000
[eval] test: accuracy=0.9651 | precision=0.9811 | recall=0.9630 | f1=0.9720 | auc=0.9954


{'experiment_summary_json': 'reports/breast_cancer_demo_run_experiment_summary.json',
 'experiment_summary_md': 'reports/breast_cancer_demo_run_experiment_summary.md'}

## 4. Inspect Config Values From Code

In [5]:
pipeline.get_section("training")


{'batch_size': 32,
 'epochs': 30,
 'learning_rate': 0.001,
 'verbose': True,
 'tensorboard_enabled': True,
 'tensorboard_log_dir': 'logs/tensorboard'}

## 5. Use a Custom or Ensemble Model

You can also inject a direct sklearn estimator or ensemble from code instead of using the config model registry.

In [6]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

ensemble = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=200)),
        ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
    ],
    voting="soft",
)

pipeline = TrainingPipeline(config_path=DEFAULT_CONFIG)
pipeline.set_dataset_frame(demo_df, target_column="target", dataset_name="breast_cancer_demo")
pipeline.set_model_instance(ensemble, model_name="voting_classifier", task_type="classification")
pipeline.set_training(verbose=True)

result = pipeline.run(run_id="breast_cancer_ensemble_demo", save_checkpoint=False)
result["metrics"]

2026-04-25 00:36:00,945 | INFO | training_pipeline | CONFIG SNAPSHOT
2026-04-25 00:36:00,945 | INFO | training_pipeline | CONFIG | section=dataset | value={"path": "data/raw/example.csv", "abnormal_path": "", "target_column": "target", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}
2026-04-25 00:36:00,945 | INFO | training_pipeline | CONFIG | section=preprocessing | value={"missing_value_strategy": "mean", "normalization": "standard", "encoding": "none", "feature_selection": false, "selected_features": []}
2026-04-25 00:36:00,945 | INFO | training_pipeline | CONFIG | section=model | value={"type": "pytorch_mlp", "task_type": "classification", "hyperparameters": {"input_dim": 2, "hidden_dim": 16, "output_dim": 2}, "name": ""}
2026-04-25 00:36:00,945 | INFO | training_pipeline | CONFIG | section=training | value={"batch_size": 8, "epochs": 5, "learning_rate": 0.001, "verbose": true, "tensorboard_enabled": true, "tensorboard_log_dir": "logs

[train] model=voting_classifier type=sklearn status=trained
[eval] validation: accuracy=0.9882 | precision=1.0000 | recall=0.9811 | f1=0.9905 | auc=1.0000
[eval] test: accuracy=0.9651 | precision=0.9636 | recall=0.9815 | f1=0.9725 | auc=0.9936


{'validation': {'accuracy': 0.9882352941176471,
  'precision': 1.0,
  'recall': 0.9811320754716981,
  'f1': 0.9904761904761905,
  'confusion_matrix': [[32, 0], [1, 52]],
  'auc': 1.0},
 'test': {'accuracy': 0.9651162790697675,
  'precision': 0.9636363636363636,
  'recall': 0.9814814814814815,
  'f1': 0.9724770642201835,
  'confusion_matrix': [[30, 2], [1, 53]],
  'auc': 0.9936342592592593}}

## 6. Save a Modified Config

This is useful when you want to turn notebook experimentation into a repeatable CLI run later.

In [ ]:
pipeline = TrainingPipeline(config_path=DEFAULT_CONFIG)
pipeline.set_model(model_type="random_forest", hyperparameters={"n_estimators": 100, "max_depth": 6})
pipeline.set_training(epochs=1, batch_size=32, verbose=False)
pipeline.save_config("configs/notebook_demo_config.json")